<a href="https://colab.research.google.com/github/Chhoulong-Banh/cyclistic-bike-share-analysis/blob/main/cyclistic_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment & Credentials Setup
# We use 'userdata' to securely pull sensitive passwords that shouldn't be hardcoded.
from google.colab import userdata
from sqlalchemy import create_engine
import os

# Securely retrieve database credentials stored in the Colab "Secrets" vault (the key icon)
# This prevents your passwords from being visible in your notebook or GitHub repository.
host = userdata.get('AZURE_HOST')
db   = userdata.get('AZURE_DB')
user = userdata.get('AZURE_USER')
password = userdata.get('AZURE_PASS')

# Create the SQLAlchemy "engine," which acts as the bridge (connector) to your Azure database.
# SSL is required by Azure for secure, encrypted connections.
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}/{db}?sslmode=require"
engine = create_engine(connection_string)

# Perform a "handshake" test to ensure the connection is successful.
try:
    with engine.connect() as conn:
        print("Successfully connected to Azure PostgreSQL!")
except Exception as e:
    # If there is an error (like a wrong password or network issue), this explains why.
    print(f"Connection failed: {e}")

In [ ]:
# Cell 2: The Diagnostic Profiler Part 1
# --- STEP 1: Environment & Authentication ---
# We mount Google Drive to access your raw 2025 Cyclistic data files.
from google.colab import drive
import os

# Mount the drive. This creates a link between your Google Drive and the Colab environment.
drive.mount('/content/drive')

# Define the central directory path for the project.
# This variable acts as a 'home base' for all subsequent operations.
BASE_DIR = '/content/drive/MyDrive/Demo/CaseStudy1-CyclisticBikes/'
RAW_DATA_PATH = os.path.join(BASE_DIR, 'OriginalDataFile-2025/')

# Validation check: Ensure the folder exists before attempting to process data.
if os.path.exists(RAW_DATA_PATH):
    print(f"Connection established! Data detected at: {RAW_DATA_PATH}")
else:
    print(f"ALERT: The folder {RAW_DATA_PATH} was not found. Check your Drive path.")

In [ ]:
# Cell 3: The Diagnostic Profiler Part 2
# --- STEP 2: The Diagnostic Profiler ---
# Objective: Scan the raw data files, infer their structure, and identify any issues.
import pandas as pd
import json
from pathlib import Path

# Load schema rules (our 'contract' for what the data should look like).
TYPE_RULES = load_type_rules(TYPE_RULES_PATH) 
# EXPLICIT_DTYPES tells pandas exactly how to read columns to avoid interpretation errors.
EXPLICIT_DTYPES = {col: eval(dtype) for col, dtype in TYPE_RULES["read_csv_explicit_dtypes"].items()}

# Generate a list of all CSV files in the data directory.
files = list(Path(RAW_DATA_PATH).glob('*.csv'))

print(f"Starting analysis of {len(files)} files...")

# Iterative Profiling: Loop through each month to "learn" its structure.
for file in files:
    # We profile the first 100 rows; this is enough to understand the schema without processing gigabytes of data.
    df = pd.read_csv(file, nrows=100, dtype=EXPLICIT_DTYPES)
    
    # Identify column data types and convert to standard SQL types for Azure.
    current_schema = {col: SQL_TYPE_MAP.get(str(df[col].dtype), 'VARCHAR(255)') for col in df.columns}
    
    # Apply custom overrides defined in our configuration (e.g., forcing lat/lng coordinates).
    for col in current_schema:
        for pattern, sql_type in OVERRIDES.items():
            if pattern in col:
                current_schema[col] = sql_type
    
    # Schema Contract Check:
    # If the file's structure differs from our master_schema, this identifies the drift.
    # [Your validation logic would follow here to flag or log discrepancies]
    print(f"Profiling complete for: {file.name}")